# Camera-ready revision experiments (E1–E5)

Trains each method once per LOSO fold and evaluates it on the **same** calibration draws under three protocols (`balanced` = paper, `natural`, `contiguous`), plus EEGNet-backbone meta-learners (E2), fine-tuned EEGNet (E3), timing (E4) and xDAWN-MDM / MDWM (E5).

Attach: the `errp_bci` dataset (**new version with `revision.py`**) and the INRIA or Coadaptation dataset. GPU on. Checkpoints are per fold, so a timed-out session can simply be re-run (with the previous output attached as `PRIOR`).

In [ ]:
import sys, os, glob, shutil
!pip -q install pyriemann

def _find_pkg():
    for c in glob.glob('/kaggle/input/**/errp_bci', recursive=True) + [os.path.join(os.getcwd(), 'errp_bci')]:
        if os.path.isfile(os.path.join(c, 'revision.py')):
            return os.path.dirname(os.path.abspath(c))
    raise FileNotFoundError('errp_bci with revision.py not found - upload the new dataset version')
_root = _find_pkg()
sys.path.insert(0, _root)
from errp_bci.config import Config
from errp_bci.revision import run_revision, collect, GROUPS
from errp_bci import revision_analysis as ra
print('errp_bci from', _root, '| device', Config.DEVICE, '| groups', GROUPS)

In [ ]:
DATASET = "coadaptation"   # "inria" | "coadaptation"
SEEDS   = [42]             # one seed: the paper shows seed spread <= 0.013
GROUPS_TO_RUN = None       # None = all 10 groups, or e.g. ["EEGNet", "Reptile"]
BATCH   = (8, 16)           # first group: subjects 0-7; second session: (8, 16)
DATASET_ROOT = None        # None = auto-detect below; or set the path by hand
PRIOR   = None             # path to an earlier run's Revision_<dataset> folder to resume from

def _find_data(dataset):
    """Locate the dataset under /kaggle/input by its contents, not its name."""
    if dataset == "inria":   # folder holding TrainLabels.csv + train/
        hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/TrainLabels.csv', recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(p), 'train'))]
    else:                    # folder holding s*/s*_*.set
        hits = sorted({os.path.dirname(os.path.dirname(p))
                       for p in glob.glob('/kaggle/input/**/s*/s*_*.set', recursive=True)})
    if not hits:
        raise FileNotFoundError(f'{dataset} data not found under /kaggle/input - attach it or set DATASET_ROOT')
    print(f'{dataset} data ->', hits[0], f'({len(hits)} candidate(s))')
    return hits[0]

DATASET_ROOT = DATASET_ROOT or _find_data(DATASET)

OUT = f'/kaggle/working/Revision_{DATASET}'
if PRIOR and os.path.isdir(os.path.join(PRIOR, 'ckpt')):
    os.makedirs(os.path.join(OUT, 'ckpt'), exist_ok=True)
    for f in glob.glob(os.path.join(PRIOR, 'ckpt', '*.json')):
        shutil.copy(f, os.path.join(OUT, 'ckpt'))
    print('seeded', len(os.listdir(os.path.join(OUT, 'ckpt'))), 'checkpoints from', PRIOR)

In [ ]:
out = run_revision(DATASET, seeds=SEEDS, groups=GROUPS_TO_RUN, batch=BATCH,
                   dataset_root=DATASET_ROOT, out_dir=OUT)

In [ ]:
# Tables (preliminary if only part of the folds are done)
res = ra.report(OUT)

In [ ]:
# Zip the whole output folder for download (draws.csv, timing.csv, burden.csv, ckpt/)
shutil.make_archive(f'/kaggle/working/Revision_{DATASET}', 'zip', OUT)
print('download /kaggle/working/Revision_' + DATASET + '.zip')